In [2]:
import pandas as pd
from plotnine import *
import os
import json

colors_dict = {
"SCoNE":"#2f4b7c",
"MVBC":"#665191",
"RGWAS":"#a05195",  
"C-NMF":"#d45087",  
"C-CoNE":"#f95d6a",  
"G-NMF":"#ff7c43",  
"G-CoNE":"#ffa600",  
"HNMF":"#2ca02c",  
"SCoNE(Fro)":"#1f77b4",
"CoNE":"#17becf"
}

In [6]:
output_dir = '/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output'
os.makedirs(f'{output_dir}/figures',exist_ok=True)

# Assess runs over correlation
noise=0, sparsity=0, weight=0.5

In [ ]:
def graph_plots(summary,variable_name,x_axis_name,y_axis_name,output_path,facet_by_factor_matrix=True,comparator_method_plot=True):
    # variable name is true var name, x-axis name is specific name for x-axis label (like if need latex)
    # if facet_by_factor_matrix is True, facet_wrap by factor matrix else don't
    # baseline methods
    p = (
        ggplot(summary[summary['run_name'].isin(['G-NMF','C-NMF','G-CoNE','C-CoNE','CoNE','SCoNE','SCoNE(Fro)','HNMF'])], aes(variable_name, "mean", color='run_name', group='run_name'))
        + geom_point(size=2)
        + geom_line()
        + geom_errorbar(aes(ymin="ymin",ymax="ymax"),width=0.05)
        + scale_color_manual(values=colors_dict)
        + labs(y=y_axis_name,x=x_axis_name,color="")
        + theme_bw()
    )
    if facet_by_factor_matrix:
        p = p + facet_wrap('factor_matrix', nrow=1) + theme(legend_title=element_text(size=9),figure_size=(8,4),legend_position='top')
    else:
        p = p + theme(legend_title=element_text(size=9),figure_size=(4,4),legend_position='top') + ylim(0.5,0.9)
    p.save(f'{output_path}_baseline.jpeg',dpi=300)

    # comparator methods
    if comparator_method_plot:
        p = (
            ggplot(summary[summary['run_name'].isin(['SCoNE','RGWAS','MVBC','HNMF'])], aes(variable_name, "mean", color='run_name', group='run_name'))
            + geom_point(size=2)
            + geom_line()
            + geom_errorbar(aes(ymin="ymin",ymax="ymax"),width=0.05)
            + scale_color_manual(values=colors_dict)
            + labs(y=y_axis_name,x=x_axis_name,color="")
            + theme_bw()
            + theme(legend_title=element_text(size=9),figure_size=(8,4),legend_position='top')
        )
        if facet_by_factor_matrix:
            p = p + facet_wrap('factor_matrix', nrow=1) + theme(legend_title=element_text(size=9),figure_size=(8,4),legend_position='top')
        else:
            p = p + theme(legend_title=element_text(size=9),figure_size=(4,4),legend_position='top')
        p.save(f'{output_path}_comparator.jpeg',dpi=300)

In [18]:
def simple_graph(summary,variable_name,x_axis_name,y_axis_name,output_path):
    p = (
    ggplot(summary[summary['run_name']=='SCoNE'], aes(variable_name, "mean", color='factor_matrix', group='factor_matrix'))
    + geom_point(size=2)
    + geom_line()
    + geom_errorbar(aes(ymin="ymin",ymax="ymax"),width=0.05)
    + scale_color_brewer(type="qual", palette="Paired")
    + labs(y=y_axis_name,x=x_axis_name,color="")
    + theme_bw()
    + theme(legend_title=element_text(size=9),figure_size=(4,4),legend_position='top')
    )
    p.save(f'{output_path}_baseline.pdf',dpi=300)

In [ ]:
# visualize a loss
with open('/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/models/loss_function_lambdaval_0_noise_0.5_init_0.json')

In [20]:
# lowest loss or highest ll for RGWAS
def get_loss_or_likelihood(row):
    if row.run_name != 'MVBC': 
        with open(f'{row.out_path}_loss_function.json','r') as f:
            loss_function = json.load(f)
        if row.run_name == 'RGWAS':
            loss = - loss_function['ll'][-1]
        else:
            loss = loss_function['total_loss'][-1]
    else:
        loss = 0
    return loss

In [21]:
run_plan = pd.read_csv('/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/models/rho_run_plan.csv')
run_plan['loss'] = run_plan.apply(lambda row: get_loss_or_likelihood(row), axis=1)

In [ ]:
from test_reconstruction import get_summary_df, is_sparse
df = pd.read_csv('/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/rho_results.csv')
df['is_sparse'] = df['run_name'].apply(lambda x: is_sparse(x))

# TUNING: choose optimal sparsity parameters
# choose optimal sparsity based on avg cosine sim for W
tuning_df = df[(df['split']=='tune')&(df['factor_matrix'].isin(['W_C','W_G']))].copy()
tuning_df = tuning_df.groupby(['job_id','run_name','rho','rank','lambda_val'])['sim'].mean().reset_index()
best_df = tuning_df.loc[tuning_df.groupby(['run_name','rho'])['sim'].idxmax()]

# subset df by optimal sparsity params
df = df[(df['is_sparse'] & df['job_id'].isin(best_df['job_id'].unique()))|(~df['is_sparse'])]

# get summary for best params for split train
summary_sim = get_summary_df(df,["run_name", "factor_matrix", "rho","rank"],"sim")
# subset to W_C since rel_error recordings for all factor matrices are duplicates (don't differ by factor matrix)
summary_relerror_G = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "rho","rank"],"rel_error_G") 
summary_relerror_C = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "rho","rank"],"rel_error_C")

assert summary_sim[(summary_sim['run_name']=='MVBC')&(summary_sim['std']!=0)].shape[0] == 0

simple_graph(summary_sim,"rho",x_axis_name=r"$\rho$",y_axis_name="Similarity",output_path=f'{output_dir}/figures/rho')

/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 4 x 4 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/figures/rho_baseline.pdf


# Assess runs over ZU weight

In [20]:
df = pd.read_csv('/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/ZU_weight_results.csv')
df['is_sparse'] = df['run_name'].apply(lambda x: is_sparse(x))

# TUNING: choose optimal sparsity parameters
# choose optimal sparsity based on avg cosine sim for W
tuning_df = df[(df['split']=='tune')&(df['factor_matrix'].isin(['W_C','W_G']))].copy()
tuning_df = tuning_df.groupby(['job_id','run_name','ZU_weight','rank','lambda_val'])['sim'].mean().reset_index()
best_df = tuning_df.loc[tuning_df.groupby(['run_name','ZU_weight'])['sim'].idxmax()]

# subset df by optimal sparsity params
df = df[(df['is_sparse'] & df['job_id'].isin(best_df['job_id'].unique()))|(~df['is_sparse'])]

# get summary for best params for split train
summary_sim = get_summary_df(df,["run_name", "factor_matrix", "ZU_weight","rank"],"sim")
# subset to W_C since rel_error recordings for all factor matrices are duplicates (don't differ by factor matrix)
summary_relerror_G = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "ZU_weight","rank"],"rel_error_G") 
summary_relerror_C = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "ZU_weight","rank"],"rel_error_C")
simple_graph(summary_sim,"ZU_weight",x_axis_name=r"$zu_{weight}$",y_axis_name="Similarity",output_path=f'{output_dir}/figures/zu_weight')


/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 4 x 4 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/figures/zu_weight_baseline.pdf


# Assess runs over sparsity

In [21]:
from test_reconstruction import get_summary_df, is_sparse
df = pd.read_csv('/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/sparsity_results.csv')
df['is_sparse'] = df['run_name'].apply(lambda x: is_sparse(x))

# TUNING: choose optimal sparsity parameters
# choose optimal sparsity based on avg cosine sim for W
tuning_df = df[(df['split']=='tune')&(df['factor_matrix'].isin(['W_C','W_G']))].copy()
tuning_df = tuning_df.groupby(['job_id','run_name','sparsity','rank','lambda_val'])['sim'].mean().reset_index()
best_df = tuning_df.loc[tuning_df.groupby(['run_name','sparsity'])['sim'].idxmax()]

# subset df by optimal sparsity params
df = df[(df['is_sparse'] & df['job_id'].isin(best_df['job_id'].unique()))|(~df['is_sparse'])]

# get summary for best params for split train
summary_sim = get_summary_df(df,["run_name", "factor_matrix", "sparsity","rank"],"sim")
# subset to W_C since rel_error recordings for all factor matrices are duplicates (don't differ by factor matrix)
summary_relerror_G = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "sparsity","rank"],"rel_error_G") 
summary_relerror_C = get_summary_df(df[df['factor_matrix']=='W_C'],["run_name", "lambda_val", "sparsity","rank"],"rel_error_C")

assert summary_sim[(summary_sim['run_name']=='MVBC')&(summary_sim['std']!=0)].shape[0] == 0
simple_graph(summary_sim,"sparsity",x_axis_name=r"$sparsity$",y_axis_name="Similarity",output_path=f'{output_dir}/figures/sparsity')


/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:587: PlotnineWarning: Saving 4 x 4 in image.
/gpfs/commons/home/anewbury/miniconda/envs/jupyter/lib/python3.9/site-packages/plotnine/ggplot.py:588: PlotnineWarning: Filename: /gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/output/figures/sparsity_baseline.pdf
